# Perovskites FMQA / FMQAOA 結果可視化ノートブック (`plot_results.ipynb`)

SSHサーバー上で `python main.py` を実行して得られた出力フォルダ（例: `results/bbo_low_budget_sweep`, `results/penalty_sweep_bbo`）をローカルにコピーし、
1. **統計サマリ表（中央値・IQR・Spearman順位相関 $\rho$・Wilcoxon符号付順位検定）**
2. **BBO収束曲線（Best Objective / Normalized Regret の中央値＋IQR帯、FM厳密解 `Exact-FM` 含む）**
3. **FMサロゲート外挿精度の推移（未評価候補に対する Spearman $\rho_t$ と FM最小解の真値ランク）**
4. **サンプラー診断比較（FM最小解捕捉率・実行可能解率・最終Regret）**
5. **Seed別の最終Regret比較・ペア差分プロット**
6. **FMQAのペナルティ係数 $\alpha$ 依存性プロット（対数軸 $\alpha$ vs. 実行可能解率・FM最小解捕捉率・Regret / FM Gap ＋ `XY-FMQAOA` & `Exact-FM` 基準線）**
7. **少ショット・少Sweeps予算 ($n = \mathrm{reads} = \mathrm{shots}$) 依存性プロット（＋ `Exact-FM` 基準線）**

を表示・保存（`pdf/`, `svg/`, `png/`）します（すべての凡例はプロット枠外に配置）。

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# =====================================================================
# 読み込む結果ディレクトリの指定（サーバーから取得したフォルダ名に変更してください）
# =====================================================================
RUN_DIR = Path("results/bbo_low_budget_sweep")  # または Path("results/penalty_sweep_bbo")

CSV_DIR = RUN_DIR / "csv"
JSON_DIR = RUN_DIR / "json"
for fmt in ("pdf", "svg", "png"):
    (RUN_DIR / fmt).mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "font.size": 11,
    "axes.titlesize": 12,
    "axes.labelsize": 11,
    "legend.fontsize": 9.5,
    "figure.dpi": 130,
})

PALETTE = {
    "Exact-FM":                       {"color": "#111111", "marker": "P", "ls": "-.", "label": "Exact-FM (Exact FM Min)"},
    "TunedPenalty-FMQA":              {"color": "#1f77b4", "marker": "o", "ls": "-",  "label": r"TunedPenalty-FMQA ($\alpha=1.0$)"},
    "LargePenalty-FMQA":              {"color": "#ff7f0e", "marker": "s", "ls": "--", "label": r"LargePenalty-FMQA ($\alpha=100$)"},
    "Penalty-FMQAOA":                 {"color": "#d62728", "marker": "^", "ls": ":",  "label": r"Penalty-FMQAOA ($\alpha=5$)"},
    "XY-FMQAOA":                      {"color": "#2ca02c", "marker": "D", "ls": "-",  "label": r"XY-FMQAOA ($\lambda=0$)"},
    "XY-FMQAOA(p=1,mean,legacy9)":    {"color": "#8c564b", "marker": "v", "ls": ":",  "label": r"XY ($p=1$, Mean, 9pt)"},
    "XY-FMQAOA(p=1,mean,wide49)":     {"color": "#9467bd", "marker": "<", "ls": "--", "label": r"XY ($p=1$, Mean, 49pt)"},
    "XY-FMQAOA(p=1,cvar,wide49)":     {"color": "#17becf", "marker": ">", "ls": "-.", "label": r"XY ($p=1$, $\mathrm{CVaR}_{0.1}$, 49pt)"},
    "XY-FMQAOA(p=3,mean,powell49)":   {"color": "#bcbd22", "marker": "p", "ls": "--", "label": r"XY ($p=3$, Mean, 49pt)"},
    "XY-FMQAOA(p=3,cvar,powell49)":   {"color": "#2ca02c", "marker": "D", "ls": "-",  "label": r"XY ($p=3$, $\mathrm{CVaR}_{0.1}$, 49pt)"},
    "XY-FMQAOA(p=3,cvar,powell1024)": {"color": "#006d2c", "marker": "*", "ls": "-",  "label": r"XY ($p=3$, $\mathrm{CVaR}_{0.1}$, 1024pt)"},
}

_cmap = plt.get_cmap("tab10")
def style_of(method: str, idx: int = 0, total: int = 1) -> dict:
    if method in PALETTE:
        return PALETTE[method]
    c = _cmap(idx % 10)
    ls = "--" if "LargePenalty" in method else ("-" if "XY" in method else "-.")
    mk = "D" if "XY" in method else ("s" if "LargePenalty" in method else "o")
    return {"color": c, "marker": mk, "ls": ls, "label": method}

def place_legend_outside_top(fig: plt.Figure, ax_for_handles: plt.Axes, max_cols: int = 4) -> None:
    """プロット枠の上部外側に凡例を配置し、データ線との重なりを防ぐ。"""
    handles, labels = ax_for_handles.get_legend_handles_labels()
    if not handles:
        return
    ncol = min(max_cols, max(1, len(labels)))
    fig.legend(
        handles, labels,
        loc="lower center",
        bbox_to_anchor=(0.5, 1.01),
        ncol=ncol,
        frameon=True,
    )

def save_fig(fig: plt.Figure, stem: str) -> None:
    for fmt in ("pdf", "svg", "png"):
        fig.savefig(RUN_DIR / fmt / f"{stem}.{fmt}", bbox_inches="tight", dpi=300)
    print(f"Saved figure: {stem}.{{pdf,svg,png}} in {RUN_DIR}")

config_used = json.loads((JSON_DIR / "config_used.json").read_text(encoding="utf-8"))
summary_data = json.loads((JSON_DIR / "summary.json").read_text(encoding="utf-8"))
df_steps = pd.read_csv(CSV_DIR / "step_history.csv")
df_seeds = pd.read_csv(CSV_DIR / "seed_summary.csv")

print(f"Loaded run from: {RUN_DIR.resolve()}")
print(f"Mode: {config_used.get('mode', 'bbo')} | Seeds ({len(config_used['seeds'])}): {config_used['seeds']}")
print(f"Methods ({len(config_used['methods'])}): {config_used['methods']}")

## 1. 統計サマリ表（中央値 [Q25, Q75]）とペア検定（Wilcoxon signed-rank test）

In [ ]:
rows = []
for m in config_used["methods"]:
    ms = summary_data["methods"][m]
    def fmt_iqr(key, digits=4):
        d = ms.get(key, {})
        if d.get("median") is None or np.isnan(d.get("median", np.nan)):
            return "N/A"
        return f"{d['median']:.{digits}f} [{d['q25']:.{digits}f}, {d['q75']:.{digits}f}]"
    rows.append({
        "Method": m,
        "Reads/Shots": ms.get("sample_budget"),
        "SA Sweeps": ms.get("sa_sweeps"),
        "Penalty alpha": ms.get("penalty_alpha"),
        "Final Best (eV) [IQR]": fmt_iqr("final_best_objective", 4),
        "Normalized Regret [IQR]": fmt_iqr("final_normalized_regret", 4),
        "FM-Min Hit Rate [IQR]": fmt_iqr("mean_fm_minimum_hit", 3),
        "Spearman Rho [IQR]": fmt_iqr("mean_unseen_spearman_rho", 3),
        "Raw Feasible Rate [IQR]": fmt_iqr("mean_raw_feasible_rate", 3),
        "Runtime (s) [IQR]": fmt_iqr("total_elapsed_sec", 2),
    })

df_summary_table = pd.DataFrame(rows)
display(df_summary_table)

if summary_data.get("paired_tests"):
    df_tests = pd.DataFrame([
        {
            "Comparison": t["pair"],
            "Metric": t["metric"],
            "Diff Median [Q25, Q75]": f"{t['diff_median_iqr']['median']:.4f} [{t['diff_median_iqr']['q25']:.4f}, {t['diff_median_iqr']['q75']:.4f}]",
            "p-value (raw)": f"{t['p_value_raw']:.4g}",
            "p-value (Bonferroni)": f"{t['p_value_bonferroni']:.4g}",
        }
        for t in summary_data["paired_tests"]
    ])
    display(df_tests)

## 2. BBO収束曲線（Best Objective & Normalized Regret vs. 評価予算ステップ）
- 実線：**Seed 中央値 (Median)**
- 塗りつぶし帯：**四分位範囲 (IQR: 25%〜75%パーセンタイル)**

In [ ]:
methods = config_used["methods"]
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.6), sharex=True)

for idx, method in enumerate(methods):
    sub = df_steps[df_steps["method"] == method]
    grouped = sub.groupby("budget_index")
    x = np.array(sorted(sub["budget_index"].unique()))
    st = style_of(method, idx, len(methods))

    for ax, col, ylabel, title in [
        (axes[0], "best_objective_so_far", "Best HSE06 Bandgap Distance (eV)", "Best Objective Convergence"),
        (axes[1], "normalized_regret_so_far", "Normalized Regret $R$", "Normalized Regret Convergence"),
    ]:
        q25 = grouped[col].quantile(0.25).values
        q50 = grouped[col].quantile(0.50).values
        q75 = grouped[col].quantile(0.75).values
        ax.plot(x, q50, color=st["color"], ls=st["ls"], marker=st["marker"], ms=4.5, lw=2.0, label=st["label"])
        ax.fill_between(x, q25, q75, color=st["color"], alpha=0.14)
        ax.set_xlabel("Evaluation Budget Index ($N_{\\mathrm{init}} + t$)")
        ax.set_ylabel(ylabel)
        ax.set_title(title)
        ax.grid(True, alpha=0.3)

f_opt = config_used.get("environment", {}).get("f_opt")
if f_opt is not None:
    axes[0].axhline(f_opt, color="black", ls=":", lw=1.2, label=f"Global Optimum ({f_opt:.4f} eV)")
axes[1].axhline(0.0, color="black", ls=":", lw=1.2, label="Optimum ($R=0$)")

fig.tight_layout()
place_legend_outside_top(fig, axes[0], max_cols=3)
save_fig(fig, "bbo_convergence")
plt.show()

## 3. FMサロゲート外挿精度の推移（Spearman順位相関 $\rho_t$ と FM最小解の真値ランク）

In [ ]:
df_active = df_steps[df_steps["step"] >= 1].copy()
if "unseen_spearman_rho" in df_active.columns and df_active["unseen_spearman_rho"].notna().any():
    fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.5), sharex=True)
    for idx, method in enumerate(methods):
        sub = df_active[df_active["method"] == method]
        grouped = sub.groupby("budget_index")
        x = np.array(sorted(sub["budget_index"].unique()))
        st = style_of(method, idx, len(methods))

        for ax, col, ylabel, title in [
            (axes[0], "unseen_spearman_rho", r"Unseen Spearman Rank Correlation $\rho_t$", "FM Surrogate Rank Correlation"),
            (axes[1], "unseen_fm_min_true_rank", "True Rank of Unseen FM-Minimum (1 = Best)", "True Rank of FM Surrogate Minimum"),
        ]:
            q25 = grouped[col].quantile(0.25).values
            q50 = grouped[col].quantile(0.50).values
            q75 = grouped[col].quantile(0.75).values
            ax.plot(x, q50, color=st["color"], ls=st["ls"], marker=st["marker"], ms=4.5, lw=1.9, label=st["label"])
            ax.fill_between(x, q25, q75, color=st["color"], alpha=0.14)
            ax.set_xlabel("Evaluation Budget Index ($N_{\\mathrm{init}} + t$)")
            ax.set_ylabel(ylabel)
            ax.set_title(title)
            ax.grid(True, alpha=0.3)

    axes[0].axhline(0.0, color="gray", ls=":", lw=1.0)
    axes[1].axhline(1.0, color="black", ls=":", lw=1.2, label="Rank 1 (Best Unseen)")
    fig.tight_layout()
    place_legend_outside_top(fig, axes[0], max_cols=3)
    save_fig(fig, "surrogate_quality_progression")
    plt.show()

## 4. サンプラー診断指標の比較（FM最小解捕捉率・実行可能解率・最終Regret）

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14.5, 4.3))

diag_specs = [
    ("mean_fm_minimum_hit", "FM-Minimum Hit Rate", (0.0, 1.05)),
    ("mean_raw_feasible_rate", "Raw One-Hot Feasible Rate", (0.0, 1.05)),
    ("final_normalized_regret", "Final Normalized Regret", None),
]

methods = config_used["methods"]
x_pos = np.arange(len(methods))

for ax, (col, title, ylim) in zip(axes, diag_specs):
    meds, q25s, q75s = [], [], []
    for idx, m in enumerate(methods):
        vals = df_seeds[df_seeds["method"] == m][col].dropna().values
        q25, q50, q75 = np.percentile(vals, [25, 50, 75])
        meds.append(q50)
        q25s.append(q50 - q25)
        q75s.append(q75 - q50)
        st = style_of(m, idx, len(methods))
        ax.bar(idx, q50, width=0.55, color=st["color"], alpha=0.75, edgecolor="black", lw=1.0)
        jitter = np.linspace(-0.12, 0.12, len(vals))
        ax.scatter(np.full(len(vals), idx) + jitter, vals, color="black", s=20, zorder=4, alpha=0.75)
    ax.errorbar(x_pos, meds, yerr=[q25s, q75s], fmt="none", ecolor="black", elinewidth=1.6, capsize=4, zorder=5)
    ax.set_xticks(x_pos)
    ax.set_xticklabels(methods, rotation=30, ha="right")
    ax.set_title(title)
    if ylim is not None:
        ax.set_ylim(*ylim)
    ax.grid(True, axis="y", alpha=0.3)

fig.tight_layout()
save_fig(fig, "sampler_diagnostics_bar")
plt.show()

## 5. Seed別の最終 Normalized Regret 比較（ペアプロット）

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 4.5))
methods = config_used["methods"]
x_pos = np.arange(len(methods))

for seed in config_used["seeds"]:
    sub = df_seeds[df_seeds["seed"] == seed].set_index("method").reindex(methods)
    ax.plot(x_pos, sub["final_normalized_regret"].values, marker="o", ls="--", alpha=0.55, lw=1.2, label=f"Seed {seed}")

meds = [df_seeds[df_seeds["method"] == m]["final_normalized_regret"].median() for m in methods]
ax.plot(x_pos, meds, color="black", marker="D", lw=2.6, ms=7, label="Median across Seeds")

ax.set_xticks(x_pos)
ax.set_xticklabels(methods, rotation=25, ha="right")
ax.set_ylabel("Final Normalized Regret $R$")
ax.set_title(f"Per-Seed Paired Final Normalized Regret ({len(config_used['seeds'])} Seeds)")
ax.grid(True, alpha=0.3)
ax.legend(bbox_to_anchor=(1.02, 1.0), loc="upper left", frameon=True, ncol=1)
fig.tight_layout()
save_fig(fig, "per_seed_paired_regret")
plt.show()

## 6. FMQAのペナルティ係数 $\alpha$ 依存性プロット
`penalty_sweep.enabled=true` の場合に、横軸を正規化ペナルティ係数 $\alpha = \lambda_t / S_t$（対数スケール）として可視化し、`XY-FMQAOA` ($\lambda=0$) および `Exact-FM`（FM厳密解）の基準線を重ねます。

In [ ]:
df_sa = df_seeds[df_seeds["sampler_family"] == "SA"].copy()
unique_alphas = sorted(df_sa["penalty_alpha"].dropna().unique())

if len(unique_alphas) >= 3:
    third_col = "mean_selected_fm_gap" if config_used.get("mode") == "fixed_fm" else "final_normalized_regret"
    third_label = "Selected FM Gap (eV)" if config_used.get("mode") == "fixed_fm" else "Final Normalized Regret $R$"

    fig, axes = plt.subplots(1, 3, figsize=(15.0, 4.5), sharex=True)
    panels = [
        (axes[0], "mean_raw_feasible_rate", "Raw One-Hot Feasible Rate", "Feasibility vs. Penalty $\\alpha$", (0.0, 1.05)),
        (axes[1], "mean_fm_minimum_hit", "FM-Minimum Hit Rate", "FM-Min Capture vs. Penalty $\\alpha$", (0.0, 1.05)),
        (axes[2], third_col, third_label, f"{third_label} vs. Penalty $\\alpha$", None),
    ]

    grp = df_sa.groupby("penalty_alpha")
    alphas_arr = np.array(unique_alphas, dtype=float)

    for ax, col, ylabel, title, ylim in panels:
        q25 = np.array([grp.get_group(a)[col].dropna().quantile(0.25) for a in unique_alphas])
        q50 = np.array([grp.get_group(a)[col].dropna().quantile(0.50) for a in unique_alphas])
        q75 = np.array([grp.get_group(a)[col].dropna().quantile(0.75) for a in unique_alphas])

        ax.plot(alphas_arr, q50, color="#1f77b4", marker="o", lw=2.2, ms=6, label="FMQA (SA) Median [IQR]")
        ax.fill_between(alphas_arr, q25, q75, color="#1f77b4", alpha=0.18)

        df_xy = df_seeds[df_seeds["method"].str.startswith("XY-FMQAOA")][col].dropna()
        if len(df_xy) > 0:
            xy_q25, xy_q50, xy_q75 = np.percentile(df_xy.values, [25, 50, 75])
            ax.axhline(xy_q50, color="#2ca02c", ls="--", lw=2.0, label=r"XY-FMQAOA ($\lambda=0$) Median [IQR]")
            ax.axhspan(xy_q25, xy_q75, color="#2ca02c", alpha=0.12)

        df_ex = df_seeds[df_seeds["method"] == "Exact-FM"][col].dropna()
        if len(df_ex) > 0:
            ex_q25, ex_q50, ex_q75 = np.percentile(df_ex.values, [25, 50, 75])
            ax.axhline(ex_q50, color="#111111", ls="-.", lw=2.0, label="Exact-FM (Exact FM Min) Median [IQR]")
            ax.axhspan(ex_q25, ex_q75, color="#111111", alpha=0.10)

        ax.set_xscale("log")
        ax.set_xlabel(r"Normalized Penalty Coefficient $\alpha = \lambda_t / S_t$")
        ax.set_ylabel(ylabel)
        ax.set_title(title)
        if ylim is not None:
            ax.set_ylim(*ylim)
        ax.grid(True, which="both", alpha=0.3)

    fig.tight_layout()
    place_legend_outside_top(fig, axes[1], max_cols=3)
    save_fig(fig, "fmqa_penalty_alpha_dependence")
    plt.show()
else:
    print("ペナルティ係数スイープ（alpha >= 3点）ではないため、セクション6はスキップしました。")

## 7. 少ショット・少Sweeps予算 ($n = \mathrm{reads} = \mathrm{shots}$) 依存性プロット
`low_budget_sweep.enabled=true` の場合に、横軸をサンプル数 $n$（SA `num_reads` = QAOA `shots`）として、各手法系列の **FM最小解捕捉率 (`mean_fm_minimum_hit`)** と **最終 Normalized Regret (`final_normalized_regret`)** を比較し、`Exact-FM`（FM厳密解）の水平基準線を重ねます。

In [ ]:
df_sampled = df_seeds[df_seeds["sample_budget"] > 0].copy()
if "sample_budget" in df_sampled.columns and df_sampled["sample_budget"].nunique() >= 2:
    import re
    def extract_series(m: str) -> str:
        s = re.sub(r",?\s*(reads|shots)=\d+", "", m)
        s = s.replace("(,", "(").replace("()", "")
        return s
    df_sampled["series"] = df_sampled["method"].map(extract_series)
    series_list = list(dict.fromkeys(df_sampled["series"]))

    fig, axes = plt.subplots(1, 2, figsize=(13.0, 4.6), sharex=True)
    for idx, ser in enumerate(series_list):
        sub = df_sampled[df_sampled["series"] == ser]
        ns = sorted(sub["sample_budget"].unique())
        grp = sub.groupby("sample_budget")
        st = style_of(ser, idx, len(series_list))
        for ax, col, ylabel, title in [
            (axes[0], "mean_fm_minimum_hit", "FM-Minimum Hit Rate", "FM-Min Capture vs. Sample Budget $n$"),
            (axes[1], "final_normalized_regret", "Final Normalized Regret $R$", "Final Regret vs. Sample Budget $n$"),
        ]:
            q25 = np.array([grp.get_group(n)[col].quantile(0.25) for n in ns])
            q50 = np.array([grp.get_group(n)[col].quantile(0.50) for n in ns])
            q75 = np.array([grp.get_group(n)[col].quantile(0.75) for n in ns])
            ax.plot(ns, q50, color=st["color"], ls=st["ls"], marker=st["marker"], lw=2.0, ms=6, label=ser)
            ax.fill_between(ns, q25, q75, color=st["color"], alpha=0.14)
            ax.set_xscale("log")
            ax.set_xticks(ns)
            ax.get_xaxis().set_major_formatter(plt.ScalarFormatter())
            ax.set_xlabel("Sample Budget per Step $n$ (SA reads = QAOA shots)")
            ax.set_ylabel(ylabel)
            ax.set_title(title)
            ax.grid(True, which="both", alpha=0.3)

    df_ex = df_seeds[df_seeds["method"] == "Exact-FM"]
    if len(df_ex) > 0:
        for ax, col in [(axes[0], "mean_fm_minimum_hit"), (axes[1], "final_normalized_regret")]:
            ex_vals = df_ex[col].dropna().values
            ex_q25, ex_q50, ex_q75 = np.percentile(ex_vals, [25, 50, 75])
            ax.axhline(ex_q50, color="#111111", ls="-.", lw=2.0, label="Exact-FM (Exact FM Min)")
            ax.axhspan(ex_q25, ex_q75, color="#111111", alpha=0.10)

    axes[0].set_ylim(0.0, 1.05)
    fig.tight_layout()
    place_legend_outside_top(fig, axes[0], max_cols=3)
    save_fig(fig, "low_budget_sample_dependence")
    plt.show()
else:
    print("単一のサンプル予算 n のため、セクション7（サンプル数依存性プロット）はスキップしました。")